In [1]:
import random as rand
import matplotlib.pyplot as plt
import itertools
import pandas as pd
from BKT_pytorch import TorchBKT
import os
import joblib
import json
from ELO import Elo
import numpy as np

In [2]:
with open('../elo_variable.json', 'r') as Elo_Data:
    elo_data = json.load(Elo_Data)

print(elo_data)

{'globals': {'students': 500, 'init_skill_level': [0.0, 0.0], 'k_success': 1, 'k_fail': 0.5}, 'scenarios': [{'id': 1, 'num_tasks': 2, 'num_skills': 2, 'q_matrix': [[1, 0], [0, 1]], 'difficulty_level': [1, 1], 'depends': [-1, -1]}, {'id': 2, 'num_tasks': 2, 'num_skills': 2, 'q_matrix': [[1, 1], [1, 1]], 'difficulty_level': [0, 1], 'depends': [-1, -1]}, {'id': 3, 'num_tasks': 4, 'num_skills': 2, 'q_matrix': [[1, 0, 1, 0], [0, 1, 0, 1]], 'difficulty_level': [0, 0, 1, 1], 'depends': [-1, -1, -1, -1]}, {'id': 4, 'num_tasks': 4, 'num_skills': 2, 'q_matrix': [[1, 0, 1, 1], [0, 1, 1, 1]], 'difficulty_level': [1, 1, 0, 1], 'depends': [-1, -1, -1, -1]}, {'id': 5, 'num_tasks': 6, 'num_skills': 2, 'q_matrix': [[1, 0, 1, 1, 0, 1], [0, 1, 1, 0, 1, 1]], 'difficulty_level': [0, 0, 0, 1, 1, 1], 'depends': [-1, -1, -1, -1, -1, -1]}, {'id': 6, 'num_tasks': 6, 'num_skills': 2, 'q_matrix': [[1, 0, 1, 0, 1, 0], [0, 1, 0, 1, 0, 1]], 'difficulty_level': [0, 0, 1, 1, 2, 2], 'depends': [-1, -1, -1, -1, -1, -1]}

In [3]:
def load_training_data(scenario, split):
    base_path = "../Simulated_Data"
    
    filename = f"scenario_{scenario}_{split}_data.csv"
    filepath = os.path.join(base_path, filename)
    
    return pd.read_csv(filepath)

In [4]:
all_data = {}

level_skill   = [] 
mastery_level = 1.5
i = 0

global_values = elo_data["globals"]
scenarios = elo_data["scenarios"]

students          = global_values["students"]
init_skill_level  = np.array(global_values["init_skill_level"])
k_success         = global_values["k_success"]

In [5]:
for scenario in scenarios:
    scenario_id    = scenario["id"]
    num_tasks      = scenario["num_tasks"]
    difficulty_level = np.array(scenario["difficulty_level"])
    q_matrix       = np.array(scenario["q_matrix"])
    num_skills = scenario["num_skills"]
    train_df = load_training_data(scenario_id, "train")
    test_df = load_training_data(scenario_id, "test")

    print(f"Scenario {scenario_id}")

    model = TorchBKT()
    model.fit(train_df) 
    # Perform evalution
    training_auc, train_acc = model.score(df=train_df)
    print(f"Scenario {scenario_id} train AUC, {training_auc} accuracy {train_acc}")
    model.save("../Trained_Models", scenario_id)
    # print(model.params())
    test_auc, test_acc = model.score(df=test_df)
    print(f"Scenario {scenario_id} test AUC, {test_auc} accuracy {test_acc}")

    # save_bkt(model, scenario_id, num_skills)

Scenario 1
fitting skill '0' (1 tasks, 400 students)
{'forget': 0.0,
 'guess': [0.29167922995723866],
 'learn': 0.34710192680358887,
 'prior': 0.03771825134754181,
 'slip': [0.4012007652777047]}
fitting skill '1' (1 tasks, 400 students)
{'forget': 0.0,
 'guess': [0.34487880031696105],
 'learn': 0.6088098287582397,
 'prior': 0.07677838206291199,
 'slip': [0.49714492750358924]}
Scenario 1 train AUC, 0.8012610831388923 accuracy 0.6807720861172977
[Scenario 1] TorchBKT saved → ../Trained_Models/TorchBKT_scenario_1.pth
Scenario 1 test AUC, 0.8137096071770752 accuracy 0.6887966804979253
Scenario 2
fitting skill '0' (2 tasks, 400 students)
{'forget': 0.0,
 'guess': [0.2614367094999209, 0.14972694301903336],
 'learn': 0.46190914511680603,
 'prior': 0.04532243683934212,
 'slip': [0.2578832444858984, 0.6804647257203869]}
fitting skill '1' (2 tasks, 400 students)
{'forget': 0.0,
 'guess': [0.23654888448762137, 0.18794512172437647],
 'learn': 0.5522001385688782,
 'prior': 0.05606550723314285,
 'sl